In [1]:

from numpy import *
set_printoptions(legacy = '1.25')

def f(x): return sin(x)
def g(r): return 1/(1+ exp(-r))
def h(s): return s**2

functions = array([f,g,h])

def df(x): return cos(x)
def dg(r): return g(r)*(1-g(r))
def dh(s): return 2*s

derivatives = array([df,dg,dh])


In [2]:

# first version: chains

def forprop(x_in, functions):
	x = array([x_in])
	for f in functions:
		x_out = f(x_in)
		x = append(x, x_out)
		x_in = x_out
	return x

x_in = pi/4
x = forprop(x_in, functions)

print(x)


[0.78539816 0.70710678 0.66976155 0.44858053]


In [3]:

# dy/dy = 1
delta_out = 1.0


In [4]:

# first version: chains

def backprop(delta_out, x, derivatives):
	delta = array([delta_out])
	# discard last element then reverse x
	# also reverse derivatives
	for a, df in  zip(flip(x[:-1]), flip(derivatives)):
		# chain rule -- multiply by previous der
		der = df(a) * delta[0]
		delta = insert(delta, 0, der) # insert at start
	return delta
	
delta = backprop(delta_out, x, derivatives) 
print(delta)


[0.20949953 0.29627708 1.3395231  1.        ]


In [5]:

d = 3
functions, derivatives = array([h]*d), array([dh]*d)
x_in, delta_out = 5, 1

x = forprop(x_in, functions)
delta = backprop(delta_out, x, derivatives) 

print(x, delta)


[     5     25    625 390625] [625000  62500   1250      1]


In [6]:

d = 7
w = full((d,d), None)
# array indexing ranges from 0 to 6, not 1 to 7

w[3,0] = w[3,1] = w[4,1] = w[4,2] = 1
w[5,3] = w[5,4] = w[6,5] = 1

print(w)


[[None None None None None None None]
 [None None None None None None None]
 [None None None None None None None]
 [1 1 None None None None None]
 [None 1 1 None None None None]
 [None None None 1 1 None None]
 [None None None None None 1 None]]


In [7]:

activate = full(d, None)
# array indexing ranges from 0 to 6, not 1 to 7

activate[3] = lambda x,y: x+y
activate[4] = lambda y,z: max(y,z)
activate[5] = lambda a,b: a*b
print(activate)


[None None None <function <lambda> at 0x7fa8b14307c0>
 <function <lambda> at 0x7fa8b1430720>
 <function <lambda> at 0x7fa8b1430680> None]


In [8]:

def incoming(x, w, i):
	return [ w[i,j] * outgoing(x, w, j) for j in range(d) if w[i,j] ]


In [9]:

def outgoing(x, w, i):
	if x[i] != None: return x[i]
	elif activate[i]: return activate[i](*incoming(x, w, i))
	else: return None


In [10]:

# second version: networks

def forprop(x_in, w):
	d = len(w)
	x = full(d, None)
	m = len(x_in)
	x[:m] = x_in
	for i in range(m,d): x[i] = outgoing(x, w, i)
	return x
	
x_in = array([1, 2, 0])
x = forprop(x_in, w)

print(x)


[1 2 0 3 2 6 None]


In [11]:

g = full((d,d), None)
# array indexing ranges from 0 to 6, not 1 to 7

g[3,0] = lambda x,y: 1
g[3,1] = lambda x,y: 1
g[4,1] = lambda y,z: 1 if y >= z else 0
g[4,2] = lambda y,z: 1 if z > y else 0
g[5,3] = lambda a,b: b
g[5,4] = lambda a,b: a

print(g)


[[None None None None None None None]
 [None None None None None None None]
 [None None None None None None None]
 [<function <lambda> at 0x7fa8d412d300>
  <function <lambda> at 0x7fa8b1431080> None None None None None]
 [None <function <lambda> at 0x7fa8b1430e00>
  <function <lambda> at 0x7fa8b1430360> None None None None]
 [None None None <function <lambda> at 0x7fa8b1430860>
  <function <lambda> at 0x7fa8b1430d60> None None]
 [None None None None None None None]]


In [12]:

def derivative(x, delta, g, j):
	if delta[j] != None: return delta[j]
	else:
		return sum([ derivative(x, delta, g, i) * g[i,j]( *incoming(x, w, i)) * w[i,j] for i in range(d) if g[i,j] ] )


In [13]:

# second version: networks

def backprop(x, delta_out, g):
	d = len(g)
	delta = full(d, None)
	m = len(delta_out)
	delta[-m:] = delta_out
	for j in range(d-m): 
		delta[j] = derivative(x, delta, g, j)
	return delta


In [14]:

delta_out = array([1,None])
delta = backprop(x, delta_out, g)

print(delta)


[2 5 0 2 3 1 None]
